# 04 · Thinking off or thinking with a budget (stage S2)

Every run now sends the system prompt and asks for the reply `ANSWER: <answer>`. The chat template cannot switch thinking off
(`reasoning_effort="none"` does not stop it), so two ways of getting the answer are compared:

- **P** (prefill, no thinking): the answer message is opened for the model by appending it to the prompt.
- **T** (thinking with a budget): the model thinks first; after `think_budget` tokens the answer message is opened for it.

Done-when: the chosen variant parses at least 95% over the GQA and TextVQA dev items together. The study is small (20 items per
benchmark), so accuracy differences of a few points are noise.

**How to run:** Run All, no edits: it runs the one-item check (Step-3) and skips the batch (`RUN_VARIANTS = False`).
Paste the Step-3 output; after the OK, set `RUN_VARIANTS = True` and Run All again.
It needs the GPUs of `n_copies` in `configs/mvp.yaml` (2 copies: GPUs 0-3 all free). T is the slow part: allow most of
an hour. Step 5 (resolution check) is separate and off: set `RUN_RESOLUTION = True` and Run All (it does not need
`RUN_VARIANTS`).
Shut down the kernels of other notebooks first: they keep their GPUs.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_for_notebook

CONFIG = "configs/mvp.yaml"   # the config this notebook passes to setup_notebook
# GPUs for the config's n_copies (2: every gpu group, 1: the first free pair); changing it needs a kernel restart
GPU_IDS, N_COPIES = select_for_notebook(CONFIG)

In [ ]:
import math
import os
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/mvp.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
# torch is imported here because the GPU list needs it; the other third-party imports come after this cell
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("chosen GPUs, physical ids CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"chosen GPU {i} (torch index) | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")
else:
    print("no CUDA GPU visible: the model steps will fail, the rest of the notebook can still be read")


In [ ]:
import pandas as pd


In [ ]:
# Settings
N_VARIANT_ITEMS = cfg["sample_items"]  # dev items per benchmark in step 4
R = 3  # loop passes for every evaluation here
BATCH_SIZE = 1  # the reference path
N_COPY_CHECK = 4  # step 3: items compared between one copy and all copies
PREFILL_P = cfg["no_think_prefill"]  # opens the answer message and writes "ANSWER:"
VARIANTS = {  # config keys each variant changes
    "P": {"answer_prefill": PREFILL_P},
    "T": {"answer_prefill": None, "reasoning_effort": cfg["reasoning_effort"], "think_budget": cfg["think_budget"]},
}
# step 4: N_VARIANT_ITEMS items x 2 benchmarks x 2 variants
CHECK_VARIANTS = ["P"]  # step 3 one-item check; add "T" to also check thinking (slow)
RUN_VARIANTS = False  # set True after the one-item check is pasted and OK'd
# The variant used in steps 5 and 6. Step 4 fills it in; set "P" or "T" by hand when RUN_VARIANTS is False.
VARIANT = None
RUN_RESOLUTION = False  # step 5: resolution check (memory, sample, full dev run, latency pass, F09b)
RES = cfg["resolution"]["res_px"]  # long-side sizes compared in step 5
MATCH_RES = cfg["resolution"]["match_res_px"]  # DivPrune keeps each item's full token count at this size
N_RES = cfg["resolution"]["n_items"]  # dev items per benchmark in step 5
N_RES_SAMPLE = 1  # items per benchmark of the sample run before it
VARIANTS

### Step-1:- Load the model

The model and its processor are loaded once and reused by every step below. Loading is the slow part of the notebook.

In [ ]:
print("[STAGE] load model")
vlms = api.load_models(cfg, N_COPIES)
vlm = vlms[0]
tok = vlm.processor.tokenizer

### Step-2:- The prompt the model sees

One GQA dev item, rendered back from its input ids, to check where the system message lands. P appends the answer marker at the
end; T stops after the user turn. The repeated image placeholder is shown once.

In [ ]:
item = api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0]
item.question, item.answers

In [ ]:
inputs_p = api.item_inputs(vlm, item, {**cfg, **VARIANTS["P"]})
ids = inputs_p["input_ids"][0].tolist()
image_id = vlm.model.config.image_token_id
# keep only the first of each run of image placeholders
shown = [t for i, t in enumerate(ids) if not (t == image_id and i and ids[i - 1] == image_id)]
print(tok.decode(shown, skip_special_tokens=False))
print("prompt tokens", len(ids), "| image placeholders", ids.count(image_id))

In [ ]:
# each marker must be one token, or the prefill would not be what the model itself emits
prefill_ids = tok(PREFILL_P, add_special_tokens=False)["input_ids"]
assert tok.decode(prefill_ids, skip_special_tokens=False) == PREFILL_P, prefill_ids
for marker in ("<|message_model|>", "<|content_text|>"):
    assert len(tok(marker, add_special_tokens=False)["input_ids"]) == 1, f"{marker} is not one token"
assert inputs_p["input_ids"][0, -len(prefill_ids):].tolist() == prefill_ids, "the prompt of P does not end with the prefill"
print(f"✓ prompt: P ends with the prefill ({len(prefill_ids)} tokens)")

### Step-3:- One item per benchmark under both variants

The extraction rule is checked on hand-written outputs first, then one GQA and one TextVQA dev item run under P and T before any batch.
Read the raw outputs before going on.

**T is slow:** about 4 tokens/s at batch 1, and up to 288 tokens per item (two T items here).

In [ ]:
OPEN, END = "<|content_text|>", "<|end_message|>"
cases = [  # (raw output, hit_limit, prefilled, expected answer)
    (f"ANSWER: blue{END}", False, True, "blue"),
    (f"blue{END}", False, True, None),  # no marker: no answer
    ("ANSWER: woman\n\nWe need answer", True, True, "woman"),  # the answer ends at the newline
    ("ANSWER: a long answer that got cut", True, True, None),
    # thinking, then the answer message: an ANSWER line inside thinking is ignored
    (f"<|content_thinking|>maybe\nANSWER: stop{END}<|message_model|>{OPEN}ANSWER: go{END}", False, False, "go"),
]
for raw, hit, prefilled, want in cases:
    got = api.extract_answer(raw, hit, prefilled=prefilled, answer_format=cfg["answer_format"])
    assert got == want, f"extract_answer({raw[-40:]!r}, hit_limit={hit}, prefilled={prefilled}) gave {got!r}, want {want!r}"
print("✓ extraction on fixed outputs")

# P through the read path of run_evaluation: the prefill already wrote "ANSWER:", so the model writes only the answer
def reply(raw):
    """A finished generation whose output is `raw`."""
    return api.GenerationResult(token_ids=torch.tensor([]), text=raw, raw_text=raw, n_input_tokens=0, n_visual_tokens=0,
                                hit_limit=False, ttft_s=None, generate_s=0.0, peak_mem_gb=None, n_think_tokens=0,
                                n_answer_tokens=0)


p_cfg = {**cfg, **VARIANTS["P"]}
assert PREFILL_P.endswith(OPEN + cfg["answer_format"])
assert api.read_answer(reply(f" red{END}"), p_cfg) == "red"
assert api.read_answer(reply(f"yes{END}"), p_cfg) == "yes"
assert api.read_answer(reply(f"{END}"), p_cfg) is None  # empty answer
assert api.read_answer(reply(f"<|message_model|>{OPEN}ANSWER: red{END}"), {**cfg, **VARIANTS["T"]}) == "red"  # T output starts with markers
assert api.read_answer(reply(f"red{END}"), {**cfg, **VARIANTS["T"]}) is None  # thinking on: strict, no marker, no answer
print("✓ P reply: read behind the prefilled ANSWER: marker")

In [ ]:
print("[STAGE] one item per benchmark")
check_items = [item, api.load_textvqa(cfg, "dev", limit=1, root=ROOT)[0]]  # item: the GQA item of step 2
one = []
for it in check_items:
    for variant in CHECK_VARIANTS:
        changes = VARIANTS[variant]
        row = api.evaluate_item(vlm, it, {**cfg, **changes}, R)
        gold = it.answers[:3]  # GQA has 1 reference, TextVQA 10
        one.append({"benchmark": it.benchmark, "variant": variant, "question": it.question, "gold": gold,
                    **{k: row[k] for k in ("raw_output", "pred", "correct", "format_ok", "forced", "n_think_tokens",
                                           "generate_s")}})
        # printed here because the table cuts long text
        print(f"\n--- {it.benchmark} | {variant}")
        print("question:", it.question, "| gold:", gold)
        print("pred:", repr(row["pred"]), "| correct:", row["correct"], "| format_ok:", row["format_ok"],
              "| forced:", row["forced"], "| n_think_tokens:", row["n_think_tokens"],
              f"| generate_s: {row['generate_s']:.2f}")
        print("raw_output:")
        print(row["raw_output"])
one = pd.DataFrame(one)
one

In [ ]:
p_rows, t_rows = one[one.variant == "P"], one[one.variant == "T"]
assert len(p_rows) == 2 and len(t_rows) == 2 * ("T" in CHECK_VARIANTS), "expected one GQA and one TextVQA row per variant"
assert one.format_ok.all(), f"format not ok:\n{one[['benchmark', 'variant', 'format_ok', 'pred']]}"
assert (p_rows.n_think_tokens == 0).all(), "P produced thinking tokens"
for raw in p_rows.raw_output:
    assert not raw.lstrip().startswith(cfg["answer_format"]), f"P repeated the prefilled marker: {raw[:80]!r}"
print(f"✓ one item per benchmark: {CHECK_VARIANTS} format ok, P did not think, P did not repeat the marker")

#### One copy against all copies

Several copies run the items in threads. They must give the same answers as one copy, or the batch run below cannot be trusted.
Run under P, which is fast; skipped with one copy.

In [ ]:
if N_COPIES > 1:
    print("[STAGE] copies")
    half = N_COPY_CHECK // 2
    copy_items = api.load_gqa(cfg, "dev", limit=half, root=ROOT) + api.load_textvqa(cfg, "dev", limit=half, root=ROOT)
    copies_table = pd.DataFrame(api.compare_copies(vlms, copy_items, p_cfg, R))
copies_table if N_COPIES > 1 else "skipped: N_COPIES is 1"

In [ ]:
if N_COPIES > 1:
    assert len(copies_table) == N_COPY_CHECK and copies_table.same_pred.all(), "copies changed an answer; see the table"
    print(f"✓ copies: {N_COPIES} copies give the answers of one copy on {N_COPY_CHECK} items")
else:
    print("skipped: N_COPIES is 1")

### Step-4:- P against T (done-when)

The batch run: N_VARIANT_ITEMS GQA and N_VARIANT_ITEMS TextVQA dev items per variant. The parse rate over both benchmarks
together picks the variant (accuracy breaks a tie), and the done-when is that it reaches 95% (DEC3 S2). Rates and accuracies are
percentages; `accuracy` is the headline score (GQA official match, TextVQA VQA accuracy), `accuracy_lmms` the lmms-eval one
(equal for TextVQA). With 2 copies `s_per_item` includes contention between them.

In [ ]:
# One evaluation with an in-memory config copy; the file name ends with the hash of that copy
def run(bench, bench_items, variant, res_px):
    """One evaluation of `bench_items` with `variant` at `res_px`; return (summary, rows)."""
    run_cfg = {**cfg, **VARIANTS[variant], "res_px": res_px}
    name = f"{bench}_dev_n{len(bench_items)}_res{res_px}_R{R}_{variant}_b{BATCH_SIZE}_{api.config_hash(run_cfg)}.jsonl"
    out = ROOT / cfg["out_dir"] / "nb04" / name
    summary = api.run_evaluation(None, bench_items, run_cfg, R, out, resume=True, vlms=vlms, batch_size=BATCH_SIZE)
    return summary, api.read_results(out)

In [ ]:
if RUN_VARIANTS:
    print("[STAGE] variants")
    items = {
        "gqa": api.load_gqa(cfg, "dev", limit=N_VARIANT_ITEMS, root=ROOT),
        "textvqa": api.load_textvqa(cfg, "dev", limit=N_VARIANT_ITEMS, root=ROOT),
    }
    table, variant_rows = [], {}
    for variant in VARIANTS:
        for bench, bench_items in items.items():
            summary, rows = run(bench, bench_items, variant, cfg["res_px"])
            variant_rows[variant, bench] = rows
            table.append({"variant": variant, "benchmark": bench, "n": summary["n"], "accuracy": summary["accuracy"],
                          "accuracy_lmms": summary["accuracy_lmms"], "format_ok": summary["format_ok_rate"],
                          "parse_rate": summary["parse_rate"], "answer_len_ok": summary["answer_len_ok_rate"],
                          "n_think_tokens": summary["n_think_tokens_mean"], "forced": summary["forced_rate"],
                          "s_per_item": summary["generate_s_mean"]})
    table = pd.DataFrame(table)
table if RUN_VARIANTS else "skipped: RUN_VARIANTS is False"

#### How long does T think?

A forced item stops at `think_budget`, so a median close to the budget means most items are cut off.

In [ ]:
if RUN_VARIANTS:
    think = pd.DataFrame([r for (variant, _), rows in variant_rows.items() if variant == "T" for r in rows])
think.groupby("benchmark").n_think_tokens.describe() if RUN_VARIANTS else "skipped: RUN_VARIANTS is False"

#### Answers and failures

Per item: the question, each variant's parsed answer and the references. Then at most 5 outputs that gave no answer.

In [ ]:
if RUN_VARIANTS:
    per_item = pd.DataFrame([
        {"benchmark": bench, "question": it.question,
         "pred_P": variant_rows["P", bench][i]["pred"], "pred_T": variant_rows["T", bench][i]["pred"],
         "references": sorted(set(it.answers))}
        for bench, bench_items in items.items() for i, it in enumerate(bench_items)
    ])
per_item.groupby("benchmark").head(5) if RUN_VARIANTS else "skipped: RUN_VARIANTS is False"

In [ ]:
if RUN_VARIANTS:
    failures = [(v, b, r) for (v, b), rows in variant_rows.items() for r in rows if not r["parse_ok"]]
    print(f"{len(failures)} unparsed outputs; showing the last 150 chars of at most 5")
    for variant, bench, r in failures[:5]:
        print(f"  {variant} {bench} {r['item_id']}: {r['raw_output'][-150:]!r}")
else:
    print("skipped: RUN_VARIANTS is False")

In [ ]:
# Choose by parse rate over both benchmarks together, then by accuracy
if RUN_VARIANTS:
    all_rows = pd.DataFrame([{"variant": v, "parse_rate": 100.0 * r["parse_ok"], "accuracy": 100.0 * r["correct"]}
                             for (v, _), rows in variant_rows.items() for r in rows])
    combined = all_rows.groupby("variant")[["parse_rate", "accuracy"]].mean()
    VARIANT = combined.sort_values(["parse_rate", "accuracy"], ascending=False).index[0]
    print("chosen variant", VARIANT)
else:
    print("skipped: RUN_VARIANTS is False; VARIANT from the settings:", VARIANT)
combined if RUN_VARIANTS else None

In [ ]:
if RUN_VARIANTS:
    assert all(len(rows) == N_VARIANT_ITEMS for rows in variant_rows.values()), [len(r) for r in variant_rows.values()]
    parse = combined.loc[VARIANT, "parse_rate"]
    assert parse >= 95.0, f"{VARIANT}: combined parse rate {parse:.1f} is below 95"
    print(f"✓ variants: {VARIANT} parses {parse:.1f}% over GQA + TextVQA")
else:
    print("skipped: RUN_VARIANTS is False")

### Step-5:- Resolution check: a sharper image plus DivPrune against 512 px

Does a larger image help if DivPrune brings the token count back down? Both benchmarks, dev, no thinking (the answer
setup of the S5 comparison), R = 3, batch 1. Cells (`resolution` in `configs/mvp.yaml`):

- all tokens at 512, 768 and 1024 px;
- DivPrune at 768 and 1024 px keeping, **per item**, exactly that item's full token count at 512 px;
- DivPrune at 1024 px with k = 25% (a fixed-k sanity row).

`resize_long_side` never upscales: an image no larger than a size is sent as it is, so for it the cells above that size
see the same input. Finished rows are kept (resumable); the all-token 512 px cell is read from the S5 full-dev files
when they are there, accuracy and latency alike, so its timing comes from that earlier session.

In [ ]:
RES_DIR = ROOT / cfg["out_dir"] / "nb04" / "resolution"
RES_LATENCY_DIR = ROOT / cfg["out_dir"] / "nb04" / "resolution_latency"
S5_DIR = ROOT / cfg["out_dir"] / cfg["resolution"]["reuse_dir"]
S5_LATENCY_DIR = ROOT / cfg["out_dir"] / cfg["resolution"]["reuse_latency_dir"]
# the answer setup of the S5 comparison: the no-thinking prefill
assert cfg["answer_prefill"] == cfg["no_think_prefill"], cfg["answer_prefill"]
print("S5 files:", S5_DIR, "| exists:", S5_DIR.is_dir(), "| latency exists:", S5_LATENCY_DIR.is_dir())
pd.DataFrame([{"cell": c["cell"], "setting": c["setting"], **c["keys"]} for c in api.resolution_cells(cfg)])

#### Token counts per size

The full visual-token count of every item at every size (processor only). The `n_512` count is what the matched cells keep.

In [ ]:
if RUN_RESOLUTION:
    res_items = {"gqa": api.load_gqa(cfg, "dev", limit=N_RES, root=ROOT),
                 "textvqa": api.load_textvqa(cfg, "dev", limit=N_RES, root=ROOT)}
    flat_items = [it for bench_items in res_items.values() for it in bench_items]
    counts = pd.DataFrame(api.resolution_token_counts(vlm, flat_items, cfg))
    for px in RES:
        if px != MATCH_RES:
            same = int((counts[f"n_{px}"] <= counts[f"n_{MATCH_RES}"]).sum())
            print(f"{px} px: {same}/{len(counts)} items get no more tokens than at {MATCH_RES} px (small images)")
else:
    print("skipped: RUN_RESOLUTION is False")
counts.groupby("benchmark")[["long_side", *[f"n_{px}" for px in RES]]].mean() if RUN_RESOLUTION else None

#### Memory check: one item at the largest size

The item with the most tokens at 1024 px, all tokens, before any run.

In [ ]:
if RUN_RESOLUTION:
    print("[STAGE] memory check")
    largest = counts[f"n_{max(RES)}"].idxmax()
    peak = api.resolution_memory(vlm, flat_items[largest], cfg)
    for gpu, gb in peak.items():
        total = torch.cuda.get_device_properties(gpu).total_memory / 1024**3
        print(f"GPU {gpu} | peak {gb:.1f} of {total:.0f} GiB | {counts.loc[largest, f'n_{max(RES)}']} visual tokens")
    assert all(gb < torch.cuda.get_device_properties(gpu).total_memory / 1024**3 for gpu, gb in peak.items())
    print(f"✓ memory: one item at {max(RES)} px fits")
else:
    print("skipped: RUN_RESOLUTION is False")

#### Sample run: one item per benchmark

Every cell on one item per benchmark. The rows are kept: the full run does not repeat them.

In [ ]:
if RUN_RESOLUTION:
    print("[STAGE] resolution sample")
    sample_items = {b: v[:N_RES_SAMPLE] for b, v in res_items.items()}
    sample = pd.DataFrame(api.run_resolution(vlms, sample_items, cfg, RES_DIR, reuse_dir=S5_DIR))
else:
    print("skipped: RUN_RESOLUTION is False")
sample[["cell", "benchmark", "source", "res_px", "n_vis", "n_vis_full", "k", "pred", "correct", "fallback"]] if RUN_RESOLUTION else None

In [ ]:
def check_kept(rows):
    """Kept tokens of every row against the item's token counts."""
    full_count = counts.set_index(["benchmark", "item_id"])
    cells = {c["cell"]: c for c in api.resolution_cells(cfg)}
    for r in rows.itertuples():
        item = full_count.loc[(r.benchmark, r.item_id)]
        setting, keys = cells[r.cell]["setting"], cells[r.cell]["keys"]
        assert r.n_vis_full == item[f"n_{r.res_px}"], (r.cell, r.item_id, r.n_vis_full)
        if setting == "full":
            want = r.n_vis_full
        elif setting == "matched":
            want = min(item[f"n_{MATCH_RES}"], r.n_vis_full)
        else:
            want = api.n_keep_for(r.n_vis_full, keys["k"])
        assert r.n_vis == want, f"{r.cell} {r.item_id}: kept {r.n_vis}, want {want}"
        recorded = keys["k"] if setting == "fixed_k" else r.n_vis / r.n_vis_full  # matched: kept / full per item
        assert abs(r.k - recorded) < 1e-9, (r.cell, r.item_id, r.k)


if RUN_RESOLUTION:
    assert len(sample) == len(api.resolution_cells(cfg)) * sum(len(v) for v in sample_items.values())
    check_kept(sample)
    if S5_DIR.is_dir():  # the S5 hash matches today's config; stop here otherwise
        assert (sample[sample.cell == f"full_{MATCH_RES}"].source == "reused").all(), "S5 rows were not reused"
    print(f"✓ sample: matched cells keep the {MATCH_RES} px count of each item, fixed k keeps k of its own count")
else:
    print("skipped: RUN_RESOLUTION is False")

#### Full run: dev items of both benchmarks

In [ ]:
if RUN_RESOLUTION:
    print("[STAGE] resolution run")
    res_rows = api.run_resolution(vlms, res_items, cfg, RES_DIR, reuse_dir=S5_DIR)
    res_frame = pd.DataFrame(res_rows)
else:
    print("skipped: RUN_RESOLUTION is False")
res_frame.groupby(["cell", "source"]).size() if RUN_RESOLUTION else None

In [ ]:
if RUN_RESOLUTION:
    assert len(res_frame) == len(api.resolution_cells(cfg)) * len(flat_items), len(res_frame)
    check_kept(res_frame)
    if S5_DIR.is_dir():
        assert (res_frame[res_frame.cell == f"full_{MATCH_RES}"].source == "reused").all(), "S5 rows were not reused"
    print(f"✓ resolution run: every cell has every item with the expected kept count")
else:
    print("skipped: RUN_RESOLUTION is False")

#### Latency pass: batch 1, one copy

`latency_items` dev items per benchmark on the first copy only; the table takes its timing columns from this pass.

In [ ]:
if RUN_RESOLUTION:
    N_RES_LATENCY = cfg["latency_items"]
    print(f"[STAGE] latency pass, {N_RES_LATENCY} items per benchmark, 1 copy | {RES_LATENCY_DIR}")
    latency_items = {b: v[:N_RES_LATENCY] for b, v in res_items.items()}
    res_latency = api.run_resolution(vlms[:1], latency_items, cfg, RES_LATENCY_DIR, reuse_dir=S5_LATENCY_DIR)
    res_results = pd.DataFrame(api.resolution_table(res_rows, res_latency, cfg))
    print(pd.DataFrame(res_latency).groupby(["cell", "source"]).size().to_string())
else:
    print("skipped: RUN_RESOLUTION is False")
res_results[["cell", "benchmark", "n", "accuracy", "n_vis", "n_vis_full", "k", "gflops_total", "gflops_vit", "kv_cache_bytes", "latency_s", "fallback_rate"]] if RUN_RESOLUTION else None

In [ ]:
if RUN_RESOLUTION:
    assert len(res_results) == 3 * len(api.resolution_cells(cfg)), len(res_results)
    assert (res_results[res_results.benchmark != "all"].latency_n == N_RES_LATENCY).all(), "latency pass incomplete"
    assert (res_results.latency_s > 0).all() and (res_results.kv_cache_bytes > 0).all()
    print("✓ table: every cell has accuracy, cost and latency")
else:
    print("skipped: RUN_RESOLUTION is False")

#### Does a sharper image with DivPrune beat 512 px with all tokens?

DivPrune at the 512 px token count minus all tokens at 512 px, on the same items, with a paired 95% bootstrap interval.
"beats" only when the whole interval is above 0. The 1024 px lines answer the question; 768 px is shown beside it.

In [ ]:
if RUN_RESOLUTION:
    gain = pd.DataFrame([line for px in cfg["resolution"]["matched_res_px"] for line in api.resolution_gain(res_rows, cfg, px)])
else:
    print("skipped: RUN_RESOLUTION is False")
gain if RUN_RESOLUTION else None

In [ ]:
if RUN_RESOLUTION:
    api.save_table(res_results, "resolution_check")
    print(f"## <date> · S2 · resolution check (git {api.git_hash()}, dev n={N_RES} per benchmark, R={cfg['resolution']['R']}, batch 1, no thinking)")
    for r in gain.itertuples():
        print(f"- {r.pruned} vs {r.full}, {r.benchmark}: {r.acc_pruned:.1f} vs {r.acc_full:.1f}, gain {r.gain:+.1f} "
              f"[{r.ci_low:+.1f}, {r.ci_high:+.1f}] -> {r.verdict}")
    for r in res_results[res_results.benchmark == "all"].itertuples():
        print(f"- {r.cell}: acc {r.accuracy:.1f}, tokens {r.n_vis:.0f}, {r.gflops_total:.0f} GFLOPs, "
              f"KV {r.kv_cache_bytes / 1e6:.1f} MB, latency {r.latency_s:.2f} s")
    print("- Decision: res_px = <512|768|1024> because <reason>.")
else:
    print("skipped: RUN_RESOLUTION is False")

#### Figure F09b

Accuracy against mean visual tokens per image size; filled circles keep every token, hollow markers are DivPrune.

In [ ]:
if RUN_RESOLUTION:
    api.set_paper_style()
    fig_res = api.plot_resolution(res_results)
else:
    print("skipped: RUN_RESOLUTION is False")

In [ ]:
if RUN_RESOLUTION:
    res_pdf, res_png = api.save_figure(fig_res, "F09b_resolution", final=False)
    assert res_pdf.is_file() and res_png.is_file()
    print("✓ F09b draft saved")
else:
    print("skipped: RUN_RESOLUTION is False")

In [ ]:
res_table = None  # panel (c) of F02; stays None without the resolution check, F02 then has two panels
if RUN_RESOLUTION:
    textvqa_full = res_results[(res_results.setting == "full") & (res_results.benchmark == "textvqa")]
    res_table = textvqa_full[["res_px", "accuracy", "n_vis"]].rename(columns={"n_vis": "n_vis_mean"})
res_table

### Step-6:- Decision to log

Paste these lines into `docs/decisions.md`, fill in the blanks, then set `answer_prefill`, `reasoning_effort`, `think_budget` (and
`res_px` after the Step-5 lines) in `configs/mvp.yaml`.

In [ ]:
if RUN_VARIANTS:
    chosen = {**cfg, **VARIANTS[VARIANT]}
    print(f"## <date> · S2 · answer path (git {api.git_hash()})")
    print(f"- Decision: variant {VARIANT} (answer_prefill = {chosen['answer_prefill']!r}, reasoning_effort = "
          f"{chosen['reasoning_effort']}, think_budget = {chosen['think_budget'] if VARIANT == 'T' else 'off'}) because <reason>.")
    print(f"- Dev n={N_VARIANT_ITEMS} per benchmark, R={R}, res_px={cfg['res_px']}, batch {BATCH_SIZE}:")
    for r in table.itertuples():
        print(f"  - {r.variant} {r.benchmark}: acc {r.accuracy:.1f} (lmms {r.accuracy_lmms:.1f}), parse {r.parse_rate:.1f}, "
              f"format ok {r.format_ok:.1f}, forced {r.forced:.1f}, think tokens {r.n_think_tokens:.0f}, {r.s_per_item:.2f} s/item")
    if RUN_RESOLUTION:
        print("- res_px: see the resolution check lines printed in Step-5.")
    else:
        print(f"- res_px stays {cfg['res_px']}: resolution check not run.")
else:
    print("skipped: this decision needs RUN_VARIANTS = True")

### Step-7:- Figure F02

(a) parse rate and (b) accuracy of P and T on GQA and TextVQA; with `RUN_RESOLUTION`, also (c) TextVQA accuracy with all tokens per image size.
It shows the evidence behind the decision in step 6.

In [ ]:
if RUN_VARIANTS:
    api.set_paper_style()
    fig = api.plot_thinking_off(table, res_table)
else:
    print("skipped: RUN_VARIANTS must be True to plot F02")

In [ ]:
if RUN_VARIANTS:
    pdf_path, png_path = api.save_figure(fig, "F02_thinking_off", final=False)
else:
    print("skipped: F02 not saved")

In [ ]:
if RUN_VARIANTS:
    assert pdf_path.is_file() and png_path.is_file(), f"missing {pdf_path} or {png_path}"
    print("✓ F02_thinking_off draft saved as PDF and PNG (final=False, paper/ untouched)")
else:
    print("skipped: F02 not saved")

In [ ]:
print("✓ notebook 04 done | VARIANT", VARIANT)